← [Las otras dos señales](09-las-otras-dos-senales.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Cómo se mide](11-como-se-mide.ipynb) →

# 10 · Votación y decisión

Seis diagnósticos, una compuerta. Este documento explica cómo se pasa de lo uno
a lo otro, y por qué la política es la que es.



## Por qué votar y no promediar

La tentación obvia sería promediar las confianzas de ambas señales. **No se
puede**, y la razón está en el documento [04](04-factor-de-certeza.ipynb):

| | Sistema experto | Red neuronal |
| --- | --- | --- |
| Qué produce | Factor de certeza MYCIN | Probabilidad softmax |
| Suma sobre hipótesis | No suma nada en particular | Siempre 1,0 |
| Origen | Juicio del experto | Reparto interno de la red |

Un CF de 0,95 y una softmax de 0,95 **no son la misma magnitud**. Promediarlos
produciría un número sin significado. Por eso `voting.py` cuenta votos en vez de
mezclar confianzas: la confianza viaja como diagnóstico visible, pero no pondera
la decisión.



## Las reglas de la votación

De `voting.py`, tres piezas:

### 1. Mayoría estricta

```python
def majority_material(materials):
    plastic = materials.count("plastico")
    glass = materials.count("vidrio")
    if max(plastic, glass) < 2 or plastic == glass:
        return "desconocido"
    return "plastico" if plastic > glass else "vidrio"
```

Hacen falta **al menos 2 votos** de tres y que no haya empate. Dos condiciones,
no una: 1–1–abstención no basta aunque no haya empate entre las dos clases.

### 2. `desconocido` es abstención, no una tercera opción

```python
"counts_as_vote": material in ("plastico", "vidrio")
```

Cuando una señal responde `desconocido`, su voto **no se cuenta para nadie**. Se
conserva como diagnóstico visible pero no favorece a ninguna clase. Es la
diferencia entre "no sé" y "voto por la tercera opción".

### 3. Jerarquía: primaria y respaldo

```
¿mayoría de OpenAI + sistema experto?     → esa decide
   si no ↓
¿mayoría de MobileNetV2?                  → esa decide, marcada como respaldo
   si no ↓
desconocido — no se abre nada
```

Las dos señales **no valen lo mismo**. El proveedor + sistema experto es la
señal primaria; el modelo local solo se consulta si la primera no logra mayoría.



## Por qué el modelo local es respaldo

Es la decisión más discutida del diseño, y se tomó con datos.

Una primera matriz manual de 14 objetos, contando los seis votos **sin
ponderación**, coincidió con la etiqueta en 13/14 casos (92,9 %). Prometedor.

Pero una segunda matriz, ampliada a **31 pruebas físicas** con casos difíciles y
repetidos (Powerade, Sporade, perfumes, botellas de agua, Splash), bajó a 24/31
(77,4 %). El análisis mostró que **darle el mismo peso al modelo local reducía
la estabilidad**: era menos preciso y arrastraba la mayoría.

De ahí la política actual:

1. conservar los seis diagnósticos visibles;
2. decidir por mayoría del proveedor + sistema experto cuando exista;
3. usar la mayoría del modelo local **solo** como respaldo;
4. si ninguna señal tiene mayoría estricta, `desconocido`.

Las evaluaciones posteriores lo confirmaron. Sobre 68 imágenes de webcam
etiquetadas a mano:

| Señal | Exactitud |
| --- | ---: |
| OpenAI + OpenCV + sistema experto | 52/68 (76,5 %) |
| MobileNetV2 local | 43/68 (63,2 %) |



## Tres fotos, no una

La ESP32-CAM captura una ráfaga de tres por residuo. La idea es que un encuadre
malo o un reflejo puntual no arruine la decisión.

Conviene ser honesto sobre su límite: en 32 ráfagas del mismo objeto, el modelo
local dio el mismo resultado en 30 (93,8 %). Es **estable en el acierto y en el
error**. Tres fotos ayudan contra el ruido momentáneo, no contra un error
sistemático de dominio ([08](08-cambio-de-dominio.ipynb)).



## Qué pasa si el servicio no responde

Una decisión que conviene conocer porque afecta al robot en producción. Si el
servicio de visión está caído, sin cuota o inalcanzable, la ruta de Next.js
**no propaga el error**:

```json
{ "material": "desconocido", "confidence": 0, "rule_applied": "servicio de visión no disponible — ..." }
```

Devuelve una abstención. El robot no abre nada y muestra el mensaje
correspondiente. Es la misma política conservadora: ante la duda —incluso si la
duda es un fallo de infraestructura— rechazar antes que abrir la compuerta
equivocada.



## El principio de fondo

Todo el sistema está construido sobre una asimetría de costos:

> Abrir la compuerta equivocada contamina un contenedor entero. No abrir ninguna
> solo molesta a una persona.

Por eso la abstención es barata y el error es caro, y por eso hay umbrales,
mayorías estrictas y jerarquías en vez de un simple "gana el más confiado".

---

← [Las otras dos señales](09-las-otras-dos-senales.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Cómo se mide](11-como-se-mide.ipynb) →
